In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv('assembly_qc.tsv', sep='\t')
df.columns = df.columns.str.strip()
df.to_csv('assembly_qc.tsv', sep='\t', index=False)
print(df.shape)
df.head()

In [ ]:
filtered_genomes = df[df['N50'] > 100000]
print(filtered_genomes.shape)
filtered_genomes.head()

In [ ]:
# Move filtered genomes into a directory (dry-run by default)
from pathlib import Path
import shutil
import pandas as pd

outdir = Path('filtered_genomes')
outdir.mkdir(exist_ok=True)


def move_filtered_files(df_or_series, file_col='file', base_dir=Path('.'), dest_dir=outdir,
                        dry_run=True, preserve_structure=True, search_fallback=True, action='move'):
    """Move (or copy) files listed in a DataFrame column or a Series into dest_dir.

    Parameters
    - df_or_series: DataFrame (with `file_col`) or Series of file paths
    - file_col: name of column containing file paths (if passing DataFrame)
    - base_dir: Path to treat relative paths as relative to (default: notebook cwd)
    - dest_dir: destination directory
    - dry_run: if True, only print actions
    - preserve_structure: if True, try to preserve relative subdirectory structure
    - search_fallback: if True, try to locate files by name using rglob if the path doesn't exist
    - action: 'move' (default) or 'copy'

    Returns (moved, missing) lists of tuples and paths
    """
    moved = []
    missing = []

    base_dir = Path(base_dir)
    dest_dir = Path(dest_dir)

    # Accept either a Series or a DataFrame
    if isinstance(df_or_series, pd.Series):
        series = df_or_series.astype(str).str.strip().str.strip('"').str.strip("'")
    else:
        if file_col not in df_or_series.columns:
            raise KeyError(f"Column {file_col!r} not found in DataFrame")
        series = df_or_series[file_col].astype(str).str.strip().str.strip('"').str.strip("'")

    for i, s in series.items():
        # Handle missing/empty entries
        if pd.isna(s) or s == '' or s.lower() == 'nan':
            print(f"Skipping empty entry at index {i}")
            missing.append(None)
            continue

        p = Path(s).expanduser()
        p_abs = p if p.is_absolute() else (base_dir / p)
        p_abs = p_abs.resolve(strict=False)

        # Fallback: search by filename under cwd
        if not p_abs.exists() and search_fallback:
            found = next((x for x in Path.cwd().rglob(p.name)), None)
            if found:
                print(f"Fallback: using {found} for entry {i} (original: {p})")
                p_abs = found

        if p_abs.exists() and p_abs.is_file():
            # Determine destination path
            if preserve_structure:
                try:
                    rel = p_abs.relative_to(base_dir)
                    dest = dest_dir / rel
                except Exception:
                    dest = dest_dir / p_abs.name
            else:
                dest = dest_dir / p_abs.name

            dest.parent.mkdir(parents=True, exist_ok=True)

            if dry_run:
                print(f"Would {action}: {p_abs} -> {dest}")
            else:
                try:
                    if action == 'move':
                        shutil.move(str(p_abs), str(dest))
                    elif action == 'copy':
                        shutil.copy2(str(p_abs), str(dest))
                    else:
                        raise ValueError("action must be 'move' or 'copy'")
                    print(f"{action.title()}d: {p_abs} -> {dest}")
                except Exception as e:
                    print(f"Failed to {action} {p_abs} -> {dest}: {e}")
                    missing.append(str(p_abs))
                    continue

            moved.append((str(p_abs), str(dest)))
        else:
            print(f"Missing: {p_abs}")
            missing.append(str(p_abs))

    print(f"Processed {len(series)} entries; moved: {len(moved)}, missing: {len([m for m in missing if m])}")
    return moved, missing


# Run as a dry run first
#moved, missing = move_filtered_files(filtered_genomes, dry_run=True)

# Example: to actually copy files instead of moving use:
moved, missing = move_filtered_files(filtered_genomes, dry_run=False, action='copy')
